In [1]:
import sqlite3, pandas as pd
from pathlib import Path

ROOT = Path.cwd()
while not (ROOT / "data").exists() and ROOT != ROOT.parent:
    ROOT = ROOT.parent
CLEAN, DASH = ROOT / "data" / "clean", ROOT / "data" / "dashboard"
DASH.mkdir(exist_ok=True)
con = sqlite3.connect(ROOT / "data" / "upi.db")
q = lambda sql: pd.read_sql(sql, con)

rem = pd.read_csv(CLEAN / "remitter_clean.csv")
bank = pd.read_csv(CLEAN / "dim_bank.csv")

# 1. REAL: monthly trend (volume-weighted)
trend = rem.groupby("month").apply(lambda g: pd.Series({
    "approved_pct": (g.approved_pct * g.volume_mn).sum() / g.volume_mn.sum(),
    "business_decline_pct": (g.bd_pct * g.volume_mn).sum() / g.volume_mn.sum(),
    "technical_decline_pct": (g.td_pct * g.volume_mn).sum() / g.volume_mn.sum(),
    "volume_mn": g.volume_mn.sum()}), include_groups=False).round(3).reset_index()
trend.to_csv(DASH / "1_monthly_trend_REAL.csv", index=False)

# 2. REAL: bank scorecard (latest month) + impact
last = rem[rem.month == rem.month.max()].merge(bank, on="bank_key", how="left")
imp = pd.read_csv(ROOT / "data" / "impact_by_bank.csv").rename(columns={"value_rs_cr_yr": "recoverable_value_rs_cr_yr"})
score = last.merge(imp[["bank_key", "excess_td_pct", "recoverable_value_rs_cr_yr"]], on="bank_key", how="left")
score[["bank_key", "bank_type", "volume_mn", "approved_pct", "bd_pct", "td_pct", "excess_td_pct", "recoverable_value_rs_cr_yr"]].round(3)\
    .to_csv(DASH / "2_bank_scorecard_REAL.csv", index=False)

# 3. REAL: chargeback per million (Nov-25 onwards, tiny banks nahi)
q("""SELECT month, SUM(total_txns) AS txns, SUM(cb_received) AS chargebacks,
            ROUND(1e6*SUM(cb_received)/SUM(total_txns),2) AS cb_per_million
     FROM chargeback WHERE month >= '2025-11' AND is_tiny_bank = 0 GROUP BY month ORDER BY month"""
).to_csv(DASH / "3_chargeback_REAL.csv", index=False)

# 4-6. SIMULATED: hour, reason, risk ratios
q("""SELECT hour, COUNT(*) AS txns, ROUND(100.0*AVG(failure_type='Technical'),3) AS tech_fail_pct,
            ROUND(100.0*AVG(failure_type='Business'),2) AS biz_fail_pct
     FROM fact_txn GROUP BY hour ORDER BY hour""").to_csv(DASH / "4_hourly_SIMULATED.csv", index=False)
q("""SELECT failure_type, failure_reason, COUNT(*) AS fails,
            ROUND(100.0*AVG(final_outcome='Recovered on retry'),1) AS recovered_pct,
            ROUND(100.0*AVG(final_outcome='Lost'),1) AS lost_pct
     FROM fact_txn WHERE status='Failed' GROUP BY 1,2 ORDER BY fails DESC""").to_csv(DASH / "5_reasons_SIMULATED.csv", index=False)
pd.read_csv(ROOT / "data" / "risk_ratios.csv").to_csv(DASH / "6_risk_ratios_SIMULATED.csv", index=False)

for f in sorted(DASH.glob("*.csv")):
    print(f.name, len(pd.read_csv(f)), "rows")

1_monthly_trend_REAL.csv 12 rows
2_bank_scorecard_REAL.csv 50 rows
3_chargeback_REAL.csv 9 rows
4_hourly_SIMULATED.csv 24 rows
5_reasons_SIMULATED.csv 9 rows
6_risk_ratios_SIMULATED.csv 6 rows
